I am looking at the Open Parking and Camera Violations dataset from NYC Open Data, provided by the Department of Finance (last updated November 10, 2026). 

The full dataset contains 153 million rows and 19 columns, so I downloaded 50,000 rows through the API, keeping only the issue date, violation type, fine amount, county, and issuing agency. 

I will be focusing on the fine amount column (units in USD).


Data soruce: [Open Parking and Camera Violations](https://data.cityofnewyork.us/City-Government/Open-Parking-and-Camera-Violations/nc67-uf89/about_data)

In [1]:
# Data and column selection for parking violations analysis
data = "parking_violations.csv"
column = "fine_amount"

In [3]:
# Load the parking violations

import pandas as pd

violations = pd.read_csv(data)


In [4]:
# Preview the first five rows of the data

violations.head()

,issue_date,violation,fine_amount,county,issuing_agency
0,11/25/2016,DOUBLE PARKING,115.0,K,TRAFFIC
1,08/16/2019,PHTO SCHOOL ZN SPEED VIOLATION,50.0,MN,DEPARTMENT OF TRANSPORTATION
2,06/02/2017,PHTO SCHOOL ZN SPEED VIOLATION,50.0,QN,DEPARTMENT OF TRANSPORTATION
3,10/23/2016,CROSSWALK,115.0,K,TRAFFIC
4,10/23/2016,FIRE HYDRANT,115.0,K,TRAFFIC


In [ ]:
# Check the columns and types of the data

violations.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   issue_date      50000 non-null  str    
 1   violation       47509 non-null  str    
 2   fine_amount     47509 non-null  float64
 3   county          47074 non-null  str    
 4   issuing_agency  47509 non-null  str    
dtypes: float64(1), str(4)
memory usage: 1.9 MB


Data cleaning: 
Before starting the calculations, I will check my column of interst (fine_amount) for any missing values. 

Tickets that don't have a fine amount cannot be included in the calculations. 

I also want to check for "strange values" using the describe method. This will tell me whether there are any outliers that seem implausible or negative numbers that would also be implausible fine amounts. 

In [ ]:
# Check for missing values in the fine_amount column
missing_count = violations[column].isna().sum()
missing_percent = missing_count / len(violations) * 100
print(f"Missing values in {column}: {missing_count} ({missing_percent:.2f}%)")


Missing values in fine_amount: 2491 (4.98%)


In [ ]:
# Check for strange values
violations[column].describe()

count    47509.000000
mean        67.548254
std         29.711245
min         35.000000
25%         45.000000
50%         50.000000
75%         95.000000
max        515.000000
Name: fine_amount, dtype: float64

In [ ]:
# Check the violation type of the maximum fine amount to assess whether
# It is a plausible outlier or an artificially exaggerated amount.
violations[violations[column] == violations[column].max()]["violation"].value_counts()

violation
FAILURE TO DISPLAY BUS PERMIT    1
Name: count, dtype: int64

We find the cheapest fine is $35, so there are no $0 or negative fine amounts. The spread, $35 to $515 is plausible for NYC tickets, and it suggests there are no major outliers or falsely listed/exaggerated values. We see the highest value ($515) was charged due to the failure to display a bus permit. Checking this value against Violation Code 1 of the Fine Master Table, we see that it is correct. We do find there are 2491 tickets with no listed fine amount, and these will be removed. 

Data source: [Fine Master Table](https://www.nyc.gov/assets/finance/downloads/pdf/tax_and_parking_program_operations/stipulated-fines-fee-schedule.pdf)

Now, I will remove the missing values and begin the mean, median mode calculations. 

In [ ]:
# Remove tickets with no fine amount
fines = violations[column].dropna()
print(f"Number of tickets with fine amounts: {len(fines)}")

Number of tickets with fine amounts: 47509


In [ ]:
# Calculating mean, median, and mode

mean = fines.mean()
median = fines.median()
modes = fines.mode()

print(
    f"Mean fine amount: {mean:,.1f}, Median fine amount: {median:,.1f}, Most common fine (Mode): {list(modes)}"
)

Mean fine amount: 67.5, Median fine amount: 50.0, Most common fine (Mode): [50.0]


The mean fine amount is $67.5, while the most common (mode) and the median fine amount is equal. The mean is higher than the median indiciating a right skew, since a smaller number of expensive tickets raise the average. 